In [ ]:
# ==============================
# MOTION DETECTION – OPENCV
# RUNS ON GOOGLE COLAB
# ==============================

import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

# ------------------------------
# Upload video file
# ------------------------------
print("Upload a video file (mp4/avi):")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# ------------------------------
# Read video
# ------------------------------
cap = cv2.VideoCapture(video_path)

ret, frame1 = cap.read()
ret, frame2 = cap.read()

frame_count = 0

while cap.isOpened():
    if not ret:
        break

    # Frame difference
    diff = cv2.absdiff(frame1, frame2)

    # Convert to grayscale
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

    # Blur to remove noise
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Threshold
    _, thresh = cv2.threshold(blur, 20, 255, cv2.THRESH_BINARY)

    # Dilate to fill gaps
    dilated = cv2.dilate(thresh, None, iterations=3)

    # Find contours
    contours, _ = cv2.findContours(
        dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    motion_detected = False

    for contour in contours:
        if cv2.contourArea(contour) < 1000:
            continue

        motion_detected = True
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame1, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Show motion status
    status_text = "MOTION DETECTED" if motion_detected else "NO MOTION"
    cv2.putText(
        frame1,
        status_text,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    # Display frame (every few frames to avoid spam)
    if frame_count % 5 == 0:
        cv2_imshow(frame1)

    frame1 = frame2
    ret, frame2 = cap.read()
    frame_count += 1

cap.release()
print("Video processing completed.")
